# Repeated paired NONAN enrichment gate

Five repeated participant-disjoint three-fold runs compare baseline and bounded NONAN enrichment. This decision gate uses development data only: frozen NONAN and RevalExo are not read.

In [1]:
from pathlib import Path
import sys, json, gc
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
ROOT = Path.cwd().resolve(); ROOT = ROOT.parent if ROOT.name.lower() == 'notebooks' else ROOT
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from models.stroke_gait_inception import StrokeGaitInception
P = ROOT / 'data' / 'processed'; N = ROOT / 'data' / 'interim' / 'nonan_gaitprint'; D = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device:', D)
x = np.concatenate([np.load(P / 'validated_acceleration_magnitude_windows_float32.npy'), np.load(P / 'sint_maartenskliniek_external_windows_float32.npy')])
m = pd.concat([pd.read_csv(P / 'validated_window_metadata.csv'), pd.read_csv(P / 'sint_maartenskliniek_external_window_metadata.csv')], ignore_index=True)
m = m.loc[m.label.isin(['healthy', 'stroke'])].reset_index(drop=True); m['y'] = m.label.eq('stroke').astype(int); m['source'] = m.dataset_id; m['group'] = m.participant_key.astype(str)
nx = np.load(N / 'candidate_healthy_enrichment_magnitude_isolated_spike_repaired.npy', mmap_mode='r'); nm = pd.read_csv(N / 'candidate_healthy_enrichment_window_metadata.csv')
nm['y'] = 0; nm['source'] = 'nonan_gaitprint'; nm['group'] = nm.participant_key.astype(str)
cap_rng = np.random.default_rng(42); keep = np.concatenate([cap_rng.choice(v, min(64, len(v)), replace=False) for v in nm.groupby('group').indices.values()])
nx = np.asarray(nx[keep]); nm = nm.iloc[keep].reset_index(drop=True)
people = pd.concat([m[['group', 'source', 'y']].drop_duplicates(), nm[['group', 'source', 'y']].drop_duplicates()], ignore_index=True); people['stratum'] = people.source + '|' + people.y.astype(str)
rows = []; repeats = [42, 137, 202, 404, 909]
for repeat in repeats:
    folds = StratifiedKFold(3, shuffle=True, random_state=repeat)
    for fold, (trp, vap) in enumerate(folds.split(people, people.stratum)):
        trgroups, vagroups = set(people.iloc[trp].group), set(people.iloc[vap].group)
        bt, bv = m.group.isin(trgroups).to_numpy(), m.group.isin(vagroups).to_numpy(); nt, nv = nm.group.isin(trgroups).to_numpy(), nm.group.isin(vagroups).to_numpy()
        for mode in ['baseline', 'nonan_capped_10pct']:
            torch.manual_seed(repeat * 100 + fold); np.random.seed(repeat * 100 + fold)
            tx, tm = x[bt], m.loc[bt].copy()
            if mode != 'baseline': tx, tm = np.concatenate([tx, nx[nt]]), pd.concat([tm, nm.loc[nt]], ignore_index=True)
            mean, std = tx.reshape(-1, 3).mean(0), tx.reshape(-1, 3).std(0).clip(1e-4)
            z = torch.from_numpy(((tx - mean) / std).transpose(0, 2, 1).astype('float32')); y = torch.from_numpy(tm.y.to_numpy('float32'))
            keys = tm.source + '|' + tm.y.astype(str); counts = keys.value_counts(); source_cap = tm.source.map(lambda s: .1 if s == 'nonan_gaitprint' else 1.0)
            w = torch.tensor((source_cap / keys.map(counts)).to_numpy(), dtype=torch.double); gen = torch.Generator().manual_seed(repeat * 1000 + fold)
            dl = DataLoader(TensorDataset(z, y), 128, sampler=WeightedRandomSampler(w, len(w), replacement=True, generator=gen))
            net = StrokeGaitInception().to(D); opt = torch.optim.AdamW(net.parameters(), 1e-3, weight_decay=1e-4)
            for _ in range(6):
                net.train()
                for a, b in dl:
                    opt.zero_grad(); loss = torch.nn.functional.binary_cross_entropy_with_logits(net(a.to(D)), b.to(D)); loss.backward(); opt.step()
            net.eval()
            for name, ex, em in [('original', x[bv], m.loc[bv]), ('nonan_holdout', nx[nv], nm.loc[nv])]:
                with torch.inference_mode(): p = torch.sigmoid(net(torch.from_numpy(((ex - mean) / std).transpose(0, 2, 1).astype('float32')).to(D))).cpu().numpy()
                g = em.assign(p=p).groupby(['group', 'y'], as_index=False).p.mean()
                rows.append({'repeat': repeat, 'fold': fold, 'mode': mode, 'evaluation': name, 'participants': len(g), 'healthy': int((g.y == 0).sum()), 'stroke': int((g.y == 1).sum()), 'auroc': roc_auc_score(g.y, g.p) if g.y.nunique() == 2 else np.nan, 'balanced_accuracy': balanced_accuracy_score(g.y, g.p >= .5) if g.y.nunique() == 2 else np.nan, 'healthy_specificity': float((g.loc[g.y == 0, 'p'] < .5).mean())})
            del net, opt, dl, z, y; gc.collect(); torch.cuda.empty_cache() if D.type == 'cuda' else None
            print('complete', repeat, fold, mode)
out = pd.DataFrame(rows); out.to_csv(P / 'bounded_nonan_enrichment_repeated_paired.csv', index=False)
wide = out.pivot(index=['repeat', 'fold', 'evaluation'], columns='mode', values=['auroc', 'balanced_accuracy', 'healthy_specificity'])
rng = np.random.default_rng(20260902); summary = {}
for evaluation in ['original', 'nonan_holdout']:
    part = wide.xs(evaluation, level='evaluation')
    for metric in ['auroc', 'balanced_accuracy', 'healthy_specificity']:
        delta = (part[metric]['nonan_capped_10pct'] - part[metric]['baseline']).dropna().to_numpy()
        if len(delta):
            draws = np.array([rng.choice(delta, len(delta), replace=True).mean() for _ in range(10000)])
            summary[f'{evaluation}:{metric}'] = {'n_paired_units': int(len(delta)), 'mean_delta': float(delta.mean()), 'bootstrap_95_ci': [float(np.quantile(draws, .025)), float(np.quantile(draws, .975))], 'nonnegative_units': int((delta >= 0).sum())}
print(json.dumps(summary, indent=2)); (P / 'bounded_nonan_enrichment_repeated_paired_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')

device: cuda


complete 42 0 baseline


complete 42 0 nonan_capped_10pct


complete 42 1 baseline


complete 42 1 nonan_capped_10pct


...

complete 909 1 nonan_capped_10pct


complete 909 2 baseline


complete 909 2 nonan_capped_10pct


{
  "original:auroc": {
    "n_paired_units": 15,
    "mean_delta": 0.0001984876692229909,
    "bootstrap_95_ci": [
      -0.003923266099879439,
      0.004883165443205321
    ],
    "nonnegative_units": 8
  },
  "original:balanced_accuracy": {
    "n_paired_units": 15,
    "mean_delta": 0.014955437465638188,
    "bootstrap_95_ci": [
      -0.011526319321797148,
      0.038112276326921435
    ],
    "nonnegative_units": 11
  },
  "original:healthy_specificity": {
    "n_paired_units": 15,
    "mean_delta": -0.01663290387056693,
    "bootstrap_95_ci": [
      -0.09010615023093747,
      0.04799354455338572
    ],
    "nonnegative_units": 6
  },
  "nonan_holdout:healthy_specificity": {
    "n_paired_units": 15,
    "mean_delta": -0.0020892687559354176,
    "bootstrap_95_ci": [
      -0.029629629629629634,
      0.028300094966761637
    ],
    "nonnegative_units": 12
  }
}


882

Admission requires: no meaningful original-source discrimination loss; non-negative uncertainty lower bound for held-out candidate healthy specificity; and no unexplained instability. These repeated fold units are resampling evidence, not independent clinical cohorts. Frozen NONAN and RevalExo stay unavailable until one predeclared final evaluation.